# SepsisGuard-UAR — Week 1 CPU Colab workflow

Run top to bottom on **CPU**. The first cell clones `shakiful/Sepsisguard-Uar`, records the source commit, and mounts your private Google Drive. No starter ZIP upload is needed. Work happens on `/content`; archives, reports, and run metadata persist under `MyDrive/SepsisGuard-UAR`.

Real results require A=20,336 and B=20,000 records. The downloader uses a cached archive when available, then tries official downloads with the anonymous S3 fallback. B EDA is descriptive only. Preserve the original challenge target without another shift.

The explicit `SEPSISGUARD_LOCAL_VALIDATION=1` route is for labeled local checks and does not satisfy Colab/Drive acceptance criteria.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, shutil, json, uuid
LOCAL = os.environ.get('SEPSISGUARD_LOCAL_VALIDATION') == '1'
REPO_URL = 'https://github.com/shakiful/Sepsisguard-Uar.git'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
if LOCAL:
    PROJECT = Path(os.environ.get('SEPSISGUARD_PROJECT', os.getcwd())).resolve()
    DRIVE_ROOT = PROJECT/'runs/local_checkpoints'
    print('LOCAL FALLBACK: no Colab execution or Drive write is claimed.')
else:
    PROJECT = Path('/content/sepsisguard-uar')
    if PROJECT.exists():
        assert (PROJECT/'.git').is_dir(), 'Existing folder is not a Git checkout; preserve it and choose a new project path.'
        remote = subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
        assert remote.lower().rstrip('/').removesuffix('.git') == REPO_URL.lower().removesuffix('.git'), 'Unexpected repository; stop before touching existing work.'
    else:
        subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(PROJECT)],check=True)
    # Inspect the allocated runtime before requesting Drive access.
    preliminary = subprocess.check_output([sys.executable,str(PROJECT/'scripts/compute_check.py')],text=True)
    print(preliminary)
    assert not json.loads(preliminary)['cuda_available'], 'Select CPU runtime for data preparation.'
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/SepsisGuard-UAR')
os.chdir(PROJECT)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
for folder in ('starter','dataset-archives','reports','models','runs'):
    (DRIVE_ROOT/folder).mkdir(parents=True,exist_ok=True)
RUNS = PROJECT/'runs'/RUN_ID
RUNS.mkdir(parents=True,exist_ok=True)
(RUNS/'source.json').write_text(json.dumps({'repository':REPO_URL,'commit':SOURCE_COMMIT,'run_id':RUN_ID,'environment':'local_fallback' if LOCAL else 'google_colab'},indent=2)+'\n')
def run(command,name):
    print('RUN:',name,flush=True)
    r=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    (RUNS/(name+'.log')).write_text(r.stdout)
    print(r.stdout)
    r.check_returncode()
    return r
DATA = PROJECT/'data/raw'
CACHE = DRIVE_ROOT/'dataset-archives'
print('Run ID:',RUN_ID,'Source commit:',SOURCE_COMMIT)


## Install and record the observed CPU runtime

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev,download]'], 'install')
run([sys.executable, 'scripts/compute_check.py', '--output', RUNS/'compute.json'], 'compute')
compute = json.loads((RUNS/'compute.json').read_text())
assert not compute['cuda_available'], 'Select CPU runtime for Week 1, then reconnect.'
run([sys.executable, '-m', 'pip', 'freeze'], 'environment')

## Synthetic smoke test and unit/leakage tests

In [ ]:
for command, name in [
 ([sys.executable, '-m', 'sepsisguard.synthetic', '--output', 'data/synthetic', '--n-per-source', '80'], 'synthetic'),
 ([sys.executable, '-m', 'sepsisguard.eda', '--data-root', 'data/synthetic', '--output-dir', 'reports/week1_demo'], 'demo-eda'),
 ([sys.executable, '-m', 'sepsisguard.splits', '--summary', 'reports/week1_demo/patient_summary.csv', '--output', 'reports/week1_demo/split_manifest.csv'], 'demo-splits'),
 ([sys.executable, '-m', 'pytest', '-q'], 'pytest-before')]:
    run(command, name)
print('SYNTHETIC SMOKE TEST ONLY: no research estimates from these records.')

## Download the official v1.0.0 training data and verify counts

In [ ]:
# Per-run transfer concurrency; no credentials or persistent AWS settings are changed.
aws_config = RUNS / 'aws-config'
aws_config.write_text('[default]\ns3 =\n    max_concurrent_requests = 32\n')
os.environ['AWS_CONFIG_FILE'] = str(aws_config)
run([sys.executable, '-m', 'sepsisguard.download', '--data-root', DATA,
     '--archives-dir', CACHE, '--transport', 'auto'], 'real-download')
EXPECTED = {'A':20336, 'B':20000}
COUNTS = {s:len(list((DATA/f'training_set{s}').glob('*.psv'))) for s in EXPECTED}
print('Observed counts:', COUNTS)
assert COUNTS == EXPECTED, 'Stop: incomplete cohort'

## Real-data EDA: missingness, hourly/patient prevalence, record lengths

In [ ]:
run([sys.executable, '-m', 'sepsisguard.eda', '--data-root', DATA, '--output-dir', 'reports/week1'], 'real-eda')
import pandas as pd
from IPython.display import display, Image
print((PROJECT/'reports/week1/EDA_REPORT.md').read_text())
display(pd.read_csv('reports/week1/dataset_summary.csv'))
display(pd.read_csv('reports/week1/record_length_summary.csv'))
display(Image(filename='reports/week1/figures/feature_missingness.png'))
# Checkpoint EDA before proceeding.
shutil.copytree(PROJECT/'reports/week1', DRIVE_ROOT/'reports'/RUN_ID/'week1', dirs_exist_ok=True)

## Patient partitions, full-cohort audit, and leakage regression tests

In [ ]:
run([sys.executable, '-m', 'sepsisguard.splits', '--summary', 'reports/week1/patient_summary.csv',
     '--output', 'reports/week1/split_manifest.csv', '--seed', '42'], 'real-splits')
run([sys.executable, '-m', 'sepsisguard.audit', '--data-root', DATA, '--output-dir', 'reports/week1',
     '--manifest', 'reports/week1/split_manifest.csv'], 'real-audit')
run([sys.executable, '-m', 'pytest', '-q'], 'pytest-after')
manifest = pd.read_csv('reports/week1/split_manifest.csv')
assert manifest.patient_id.is_unique
assert manifest.loc[manifest.source.eq('B'), 'split'].eq('test_B').all()
print(json.loads(Path('reports/week1/DATA_AUDIT.json').read_text()))

## Preserve the completed milestone

In [ ]:
result = {'run_id':RUN_ID, 'execution_environment':'local_fallback' if LOCAL else 'google_colab',
          'drive_saved':not LOCAL, 'patient_counts':COUNTS,
          'status':'week1_data_pipeline_passed',
          'audit':json.loads(Path('reports/week1/DATA_AUDIT.json').read_text()),
          'source_version':SOURCE_COMMIT}
(RUNS/'WEEK1_RESULT.json').write_text(json.dumps(result,indent=2)+'\n')
shutil.copytree(PROJECT/'reports/week1', DRIVE_ROOT/'reports'/RUN_ID/'week1', dirs_exist_ok=True)
shutil.copytree(RUNS, DRIVE_ROOT/'runs'/RUN_ID, dirs_exist_ok=True)
print(json.dumps(result,indent=2))
print('Saved milestone:', DRIVE_ROOT/'reports'/RUN_ID/'week1')
print('Next: 02_week2_colab.ipynb; reuse this local data and private split manifest.')